# Détection & segmentation de polypes — YOLOv8 + U-Net (Kvasir-SEG)

Exécution complète sur GPU Colab (*Exécution → Modifier le type d'exécution → GPU T4*). Durée : ~20 min.

In [ ]:
# Remplacez par l'URL de votre dépôt
REPO_URL = "https://github.com/<utilisateur>/medical-imaging-yolo-unet.git"
!git clone -q $REPO_URL project
%cd project
!pip install -q -r requirements.txt

## 1. Données — Kvasir-SEG (1000 images, masques experts)

In [ ]:
!python scripts/prepare_data.py --source kvasir

## 2. Détection — YOLOv8n pré-entraîné COCO, affiné sur les polypes

In [ ]:
!python scripts/train_yolo.py --model yolov8n.pt --epochs 50 --imgsz 512 --batch 16 --device 0

## 3. Segmentation — U-Net (BCE + Dice)

In [ ]:
!python scripts/train_unet.py --epochs 40 --size 256 --batch 16 --base 32

## 4. Évaluation : détection, U-Net seule, cascade YOLO → U-Net

In [ ]:
!python scripts/evaluate.py --imgsz 512
from IPython.display import Markdown, Image, display
display(Markdown(open('results/metrics.md').read()))
display(Image('results/examples.png'))
display(Image('results/training_curves.png'))

## 5. Sauvegarder les poids

In [ ]:
from google.colab import files
files.download('weights/yolo_best.pt')
files.download('weights/unet_best.pt')